# V3: combined model of the three extension types

Compares static_stacking with combined_stacking, using static features and static + trajectory + prior-ICD representations respectively.

In the combined model, the training Pipeline of every base model refits its own trajectory screening, imputation, LASSO, TF-IDF, SVD and scaling; stacking uses 3-fold cross-fitting and the base models are not re-tuned. The embedding must not be generated on the complete development set first and then stacked.

This is an exploratory combined ablation. The separate temporal/embedding experiments must be looked at first, before judging whether the combined model improves AUROC, AUPRC, Brier, calibration and DCA. The fixed held-out set that has already been looked at is not used to choose the best extension, and there is no external validation conclusion.

**balanced simplified-compute version**: fixed base-model parameters, full keeps the 5×10 outer validation; LASSO inner 5 folds/30 λ, stacking inner 3 folds. Uses a cache separate from the parent strict version; the old predictions of the first 6 folds are not mixed in. No real patient model of the new version has been run yet.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

# Locate the project from the extension directory or the notebooks subdirectory, to avoid hard-coding a Windows drive letter.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "dn_extensions" / "settings.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("start from dn_mortality_repro_V3_extensions or its notebooks subdirectory.")
sys.path.insert(0, str(ROOT))
print("extension directory:", ROOT)


In [ ]:
# pilot is a small 1×3 outer-fold trial run; for the formal experiment set PROFILE to "full".
# Starts only when RUN_EXPERIMENT is set to True; full may take a long time.
PROFILE = "pilot"
RUN_EXPERIMENT = False
EXPERIMENTS = ['static_stacking', 'combined_stacking']
if RUN_EXPERIMENT:
    from dn_extensions.evaluation import run_evaluation
    from dn_extensions.settings import profile_options
    output, summary, comparisons = run_evaluation(profile_options(PROFILE), EXPERIMENTS)
    display(summary)
    display(comparisons)
else:
    print("model training in this notebook has not been started.")


In [ ]:
# Show the formal full results first; without results print the status, no retraining.
RESULT_PROFILE = "full"
latest = ROOT / "results" / RESULT_PROFILE / "latest.txt"
if latest.exists():
    folder = latest.parent / latest.read_text().strip()
    summary = pd.read_csv(folder / "summary.csv")
    display(summary[summary.experiment.isin(EXPERIMENTS)])
    print("detailed record:", folder / "report.md")
else:
    print("no experiment results for this mode yet. Set RESULT_PROFILE to pilot to show pilot results.")


## Experiment status and conclusions (updated automatically)

Run mode: `full`; method: balanced fixed parameters; result directory: `results/full/27464ba98305edbc`.

LASSO 5 folds/30 λ, stacking 3 folds; static and extension models use the same simplified method. No folds of the strict version were mixed in.

The fixed held-out set took no part in this experiment. Tree-model extensions need no internal stacking cross-fitting; the original fixed parameters and within-training-fold preprocessing are unchanged.

The Holm correction covers the paired comparisons currently available; corrected p-values may change when experiments are added later.

```text

       experiment  n_outer_folds  AUROC_fold_mean  AUROC_fold_sd  AUPRC_fold_mean  Brier_fold_mean  calibration_slope_fold_mean  seconds_total  AUROC_pooled_mean  AUPRC_pooled_mean  Brier_pooled_mean  cal_intercept_pooled_mean  cal_slope_pooled_mean
combined_stacking             50          0.80475        0.02738          0.61572          0.15038                      1.08695    27443.74099            0.80347            0.60377            0.15038                   -0.00846                1.05953
  static_stacking             50          0.80466        0.02657          0.61586          0.15048                      1.07477     1319.71249            0.80325            0.60386            0.15047                    0.00134                1.05015

```

Paired differences:

```text

       experiment       reference metric  mean_diff  sd_diff  se_corrected       lo      hi        t  df  p_value  k  r  p_holm_current_family
  static_stacking       static_lr  AUROC    0.00539  0.00823       0.00298 -0.00060 0.01138  1.80933  49  0.07654 10  5                    1.0
  static_stacking       static_lr  AUPRC    0.00847  0.01519       0.00550 -0.00259 0.01952  1.53947  49  0.13012 10  5                    1.0
  static_stacking       static_lr  Brier   -0.00044  0.00289       0.00105 -0.00254 0.00166 -0.41929  49  0.67683 10  5                    1.0
combined_stacking static_stacking  AUROC    0.00009  0.01052       0.00381 -0.00757 0.00774  0.02245  49  0.98218 10  5                    1.0
combined_stacking static_stacking  AUPRC   -0.00015  0.02065       0.00748 -0.01517 0.01487 -0.01986  49  0.98424 10  5                    1.0
combined_stacking static_stacking  Brier   -0.00010  0.00393       0.00142 -0.00296 0.00276 -0.06938  49  0.94497 10  5                    1.0

```

- `static_stacking` vs `static_lr`: ΔAUROC=0.0054, corrected 95% CI [-0.0006, 0.0114]; no evidence of a difference confirmed after this round of multiple-comparison correction. The Holm-corrected p-value, AUPRC, calibration and DCA should also be checked.

- `combined_stacking` vs `static_stacking`: ΔAUROC=0.0001, corrected 95% CI [-0.0076, 0.0077]; no evidence of a difference confirmed after this round of multiple-comparison correction. The Holm-corrected p-value, AUPRC, calibration and DCA should also be checked.

> Historical snapshot: the fifth repeat is now complete; refer to the "Final development-set ablation record for stacking" later in this notebook.

# Combined-extension ablation: meeting notes on the first four repeats (2026-10-14, provisional)

**Current conclusion: the combined extensions show no additional performance gain over static stacking.**

This is a snapshot from an interim look at the results, not the final report. Only the complete rep 1–4 (seeds 42, 43, 44, 45) are used, 10 folds each, 40 folds in total; all folds of rep 5 are excluded. All models are truncated to the same first four repeats, and the validation patient indices of every fold were checked to agree. Development set of 2,045 patients, 544 deaths; the fixed held-out labels were not read, and the running code and cache were not retrained or modified.

## Comparison on the same 40 folds

| Model | AUROC (40-fold mean) | AUPRC (40-fold mean) | Brier (40-fold mean, lower is better) |
|---|---:|---:|---:|
| Static LR | 0.7986 | 0.6073 | 0.1512 |
| Static RF | 0.7972 | 0.6051 | 0.1520 |
| Static XGBoost | 0.7946 | 0.6041 | 0.1527 |
| Static stacking | 0.8038 | 0.6145 | 0.1508 |
| Static + temporal features LR | 0.7942 | 0.5992 | 0.1533 |
| Static + prior-diagnosis burden LR | 0.7985 | 0.6064 | 0.1512 |
| Static + prior-diagnosis burden + ICD embedding LR | 0.7954 | 0.6044 | 0.1527 |
| Static + temporal features + prior-diagnosis representation stacking | 0.8038 | 0.6129 | 0.1510 |

Above, the metrics are computed per fold first and then averaged over the 40 folds. Stitching the OOF probabilities of the 10 folds of each repeat back into the complete development set, computing the metrics, and finally averaging over the four repeats gives another set of numbers:

| Model | AUROC | AUPRC | Brier |
|---|---:|---:|---:|
| Static stacking | 0.802404 | 0.602385 | 0.150758 |
| Static + temporal features + prior-diagnosis representation stacking | 0.802430 | 0.600443 | 0.150977 |

The two summaries answer slightly different comparisons and must not be mixed. The pooled AUROC of the combined model is only 0.000026 higher; the 40-fold mean AUROC is instead 0.000067 lower. Both show that AUROC practically does not change. The means here come from repeated cross-validation, not from a single training run, and are not 95% confidence intervals.

## Is the combined model consistently better than static stacking?

| Repeat (seed) | Static stacking AUROC | Combined stacking AUROC | Combined minus static |
|---|---:|---:|---:|
| 42 | 0.803914 | 0.802084 | -0.001830 |
| 43 | 0.800608 | 0.800264 | -0.000344 |
| 44 | 0.800782 | 0.801098 | +0.000316 |
| 45 | 0.804311 | 0.806275 | +0.001964 |

Across the four repeats the pooled AUROC rises twice and falls twice; across the 40 paired folds it rises in 25 and falls in 15. Fold size, the size of the differences and the dependence between repeats all have to be considered, so a significant gain cannot be judged just by "how many folds were won".

After the Nadeau–Bengio correction for the dependence in repeated cross-validation, combined model minus static stacking:

- AUROC mean difference −0.000067, 95% difference interval [−0.007817, +0.007683], p=0.9861.
- AUPRC mean difference −0.001629, 95% difference interval [−0.016767, +0.013509], p=0.8288.
- Brier mean difference +0.000216, 95% difference interval [−0.002511, +0.002944], p=0.8734; a positive value means slightly worse.
- With Holm correction per metric over the current 7 comparisons, the corrected p-values of these three comparisons are all 1.

These are exploratory interim intervals and tests and cannot serve as the pre-specified final confirmatory test. Failing to detect a difference does not prove that the two models are equivalent. The planned fifth repeat is kept, and the final complete results are summarised separately.

## Calibration, clinical utility and whether the features actually enter the model

Mean pooled calibration slope over the four repeats: static stacking 1.0470, combined stacking 1.0532, close to each other and both near 1; Brier did not improve. Of the 56 DCA threshold points in 0.05–0.60, the mean net benefit of the combined model is higher at 32, lower at 22 and equal at 2; the differences are very small and no uncertainty was computed, so an overall improvement in clinical utility cannot be claimed, and different thresholds cannot be treated as independent samples.

Checking the LR base models of the 40 final outer fits of the combined model: every fold selected at least 1 temporal feature, median 4 (range 1–13); every fold has a 32-dimensional ICD embedding. The 4 numeric prior-diagnosis-burden indicators are added directly after the LASSO screening and are not excluded. Temporal features take part in LASSO; diagnosis burden and embedding are appended directly, so the absence of diagnosis-burden columns in selected_numeric does not mean the model does not use them. The presence of a 32-dimensional embedding also does not mean that every dimension contributes usefully. The median number of numeric variables selected by LASSO per fold in the combined model is 24, plus the 4 appended diagnosis-burden indicators and the 32-dimensional embedding.

## How to interpret this for now

The 40-fold mean AUROC of static stacking is about 0.0052 higher than static LR, but the corrected interval still includes 0 (−0.001436 to +0.011871, p=0.1208 without multiplicity correction). It can be treated as a promising candidate model, but cannot be said to be statistically confirmed as better than LR.

The temporal-trajectory, prior-diagnosis-burden and ICD-embedding experiments are complete, but the current implementation shows no clear additional gain. Possible reasons include: the static features and existing scores already capture part of the illness information; repeated measurements of some labs within 24 hours are limited; prior diagnosis records are not equivalent to a complete out-of-hospital history; adding dimensions may increase estimation variance. These are explanations still to be tested, not causes proven by this comparison. The results apply only to the current feature construction, selection and fixed-parameter version, and do not show that all temporal models or multimodal methods are useless. The embedding here comes from structured ICD diagnoses, not clinical text.

The recorded per-fold times of the combined model over these first four repeats add up to about 6.33 hours, versus about 17.94 minutes for static stacking. Cache reuse and parallel load differed, so this is not a strictly controlled timing experiment, but the computational cost of the combined pipeline is clearly higher. For now static stacking and the LR comparator can be kept as the priority, with the final choice made after the fifth repeat is complete.

## What can be said at the meeting

> We ran ablations of stacking, temporal trajectories and prior-diagnosis representations on the same development set and the same splits. Over the first four repeats of ten folds, the mean AUROC of static stacking is about 0.804 and the combined model is still about 0.804; AUPRC and Brier did not improve either, and the interval of the paired difference includes 0. So for now the promising option is static stacking, and the temporal and diagnosis extensions bring no additional benefit so far. The fifth repeat is still running, and the final results and independent validation must be reported separately. The development-set CV numbers here cannot be compared directly with the paper's 0.780 to judge which is better, because the cohort and evaluation settings differ.

The full training will not be adjusted because of this interim look. The paper should present the ablation results as they are; later work can complete calibration, interpretability and clinical-utility evaluation around the selected candidate model, without repeatedly looking at the same development set to make the AUC go up.


# Final development-set ablation record for stacking (2026-10-14)

The planned 5 repeats × 10 folds, 50 folds in total, seeds 42–46, are complete. Development set of 2,045 patients, 544 deaths. Paired comparisons on the same validation folds; the fixed held-out labels were not read and no additional training was done. This complete record replaces the provisional conclusion from the first four repeats; the interim snapshot is kept for traceability.

**Conclusion: static stacking shows a small numerical gain over the current best single model, LR, in a consistent direction across the five repeats, but it is not statistically significant. Adding temporal trajectories and prior-diagnosis representations together shows no additional benefit.**

| Model | AUROC (50-fold mean) | AUPRC (50-fold mean) | Brier (50-fold mean, lower is better) |
|---|---:|---:|---:|
| Static LR | 0.799275 | 0.607397 | 0.150918 |
| Static RF | 0.797980 | 0.606182 | 0.151771 |
| Static XGBoost | 0.795041 | 0.605724 | 0.152496 |
| Static stacking | 0.804665 | 0.615863 | 0.150479 |
| Combined-extension stacking | 0.804750 | 0.615715 | 0.150381 |

These are computed per outer validation fold first and then averaged over the 50 folds. The other summary, "per repeat, stitch the OOF probabilities back into the complete development set, compute the metric, then average over the five repeats", gives AUROCs of: static LR 0.798852, static stacking 0.803249, combined stacking 0.803473. Do not mix the two summaries.

## Static stacking versus single models

Relative to static LR, the 50-fold mean AUROC increases by 0.005389 (about 0.54 percentage points), AUPRC increases by 0.008466 and Brier decreases by 0.000439. The mean AUROC and AUPRC of static stacking are also higher than those of the current fixed-parameter RF and XGBoost; this is not taken to mean that it beats every tuned single model.

The pooled OOF AUROC and AUPRC are higher than LR in all five repeats; AUROC is higher in 41 of the 50 folds. The same development set is reused, so these are not five independent external validation samples.

The Nadeau–Bengio corrected 95% interval of the paired AUROC difference is [−0.000596, +0.011375], p=0.076535; after Holm correction over the current 7 AUROC comparisons p=0.535748. AUPRC difference interval [−0.002585, +0.019518], p=0.130124; Brier difference interval [−0.002541, +0.001663], p=0.676835. It can therefore be described as "a small numerical gain, consistent in direction across repeated splits", but not as "significantly better than LR". The difference interval here is not a confidence interval for the AUROC of the stacking model itself.

## Does the combined extension improve stacking further?

Relative to static stacking, the combined model increases the 50-fold mean AUROC by only 0.000086 (about 0.009 percentage points), AUPRC decreases by 0.000148 and Brier decreases by 0.000099; the values are practically very close.

The 95% interval of the paired AUROC difference is [−0.007569, +0.007740], p=0.982179, Holm p=1. The pooled AUROC rises in three of the five repeats and falls in two; it rises in 30 of the 50 folds. The number of folds won cannot prove a gain, and non-significance cannot prove strict equivalence.

| seed | Static stacking − LR AUROC | Combined − static stacking AUROC |
|---|---:|---:|
| 42 | +0.003559 | -0.001830 |
| 43 | +0.005730 | -0.000344 |
| 44 | +0.004561 | +0.000316 |
| 45 | +0.004749 | +0.001964 |
| 46 | +0.003383 | +0.001015 |

## Limits for reporting and further experiments

For now static stacking can be treated as a candidate model, with LR kept as a simple and competitive comparator. For the combined model, report that "the current extensions bring no clear additional gain"; there is no need to claim an improvement for a mean difference close to zero. The numerical gain may come from complementary base-model predictions, but this metric comparison does not directly verify the mechanism.

This round is the balanced version with fixed base-model parameters and simplified inner computation, and its results are exploratory development-set results. They cannot be compared directly with the paper's 0.780 to claim that the paper is surpassed, because the cohort and evaluation differ. If the fixed held-out set has already been looked at, it cannot be used as a brand-new untouched test set for choosing extensions either.

The separate temporal, diagnosis-burden and embedding ablations have so far been evaluated only for LR. In combined stacking the LR, RF and XGBoost all received the combined extensions, but the new-feature results of RF and XGBoost have not been reported separately. "No benefit from combining" therefore cannot be generalised to "temporal features or embeddings are useless for every model". The four diagnosis-burden indicators and the ICD embedding are appended directly after the numeric LASSO screening and are not excluded by LASSO; the temporal features take part in the numeric screening.

What can be said at the meeting:

> Over the complete five repeats of ten folds, the mean AUROC of static stacking rises from 0.799 for LR to 0.805 and is higher in all five repeats, but the interval of the paired difference still includes zero, so the gain is not yet statistically confirmed. With temporal trajectories and diagnosis representations added it is still about 0.805, with no additional gain. For now static stacking is a promising candidate and LR remains an important comparator.
